### Semi-Automated Evaluation Pipeline on variations of prompts and models

In [1]:
# Setup
from dotenv import load_dotenv
load_dotenv('../../.env')
import os, sys
sys.path.insert(0, '../../libs')
sys.path.insert(0, '../../src/Traction/prompts')
sys.path.insert(0, '../../src/Traction')

from llm_factory_openai import LLMAgent
# Use _process_batch_async to process batch_messages
import asyncio
from llm_batch_process_utils import _process_batch_async
from llm_factory_openai import BatchAsyncLLMAgent
from llm_batch_process_utils import _merge_ids_with_responses

from prompt_utils import load_prompt, format_messages
from schema import PROMPT_REGISTRY
from pathlib import Path
import pandas as pd
from llm_batch_process_utils import _build_batch_messages_from_df_flexible
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score

In [7]:
# data_dir_monetary = Path('/data/home/xiong/data/Fund/CSR/Traction/output/finetuning/monetary/cv')
def load_transform_data(data_dir,task_type):    
    if task_type in ['monetary_agreement','fiscal_agreement','fiscal_stance','monetary_stance']:
        df_train_raw = pd.read_excel(data_dir / 'training_2.xlsx')
        df_test_raw = pd.read_excel(data_dir / 'testing_2.xlsx')
        
        if 'agreement' in task_type:
            # Agreement task: focus on agreement columns
            df_train = df_train_raw[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 
                                     'agreement_general', 'disagreement_areas']]
            df_test = df_test_raw[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 
                                   'agreement_general', 'disagreement_areas']]
            ground_truth_cols = ['agreement_general', 'disagreement_areas']
            column_mapping = {
                'COUNTRY': 'country',
                'YEAR': 'year',
                'STAFF_TEXT': 'staff',
                'AUTHORITY_TEXT': 'buff'
            }
        else:
            # Stance task: focus on stance columns
            # df_train = df_train_raw[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year',
            #                          'staff_current', 'staff_future', 'authority_current', 'authority_future']]
            # df_test = df_test_raw[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year',
            #                        'staff_current', 'staff_future', 'authority_current', 'authority_future']]
            # ground_truth_cols = ['staff_current', 'staff_future', 'authority_current', 'authority_future']
            
            df_train_stance = pd.DataFrame()
            for tp in ['staff', 'buff']:
                df_temp = df_train_raw[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_current'%tp, '%s_stance_future'%tp]]
                df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
                df_temp['type'] = tp
                df_train = pd.concat([df_train_stance, df_temp], ignore_index=True)
                
            df_test_stance = pd.DataFrame()
            for tp in ['staff', 'buff']:
                df_temp = df_test_raw[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_current'%tp, '%s_stance_future'%tp]]
                df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
                df_temp['type'] = tp
                df_test = pd.concat([df_test_stance, df_temp], ignore_index=True)
                
            ground_truth_cols = ['stance_current', 'stance_future']
            
            column_mapping = {
                'COUNTRY': 'country',
                'YEAR': 'year',
                'TEXT': 'text',
                'TEXT_AUTHOR': 'type'
            }
        return df_train, df_test, column_mapping, ground_truth_cols
    else:
        raise ValueError(f"Unknown task type: {task_type}")


In [8]:
def evaluate_prompt_and_model(
    prompt_key: str,
    model_name: str = 'gpt-5',
    data_dir: Path = Path('/data/home/xiong/data/Fund/CSR/Traction/output/finetuning/monetary/cv'),
    temperature: float = 1.0,
    batch_size: int = 8,
    reasoning_effort: str = 'low',
    max_completion_tokens: int = 2000,
    use_full_dataset: bool = True,
    return_dataframe: bool = False
):
    """
    High-level function to evaluate different prompt strategies and models.
    
    Parameters:
    -----------
    prompt_key : str
        Key from PROMPT_REGISTRY (e.g., 'monetary_agreement_simple', 'fiscal_stance_few_shot')
    model_name : str
        OpenAI model name (e.g., 'gpt-5', 'gpt-5-nano')
    temperature : float
        Temperature for LLM generation (default: 1.0)
    batch_size : int
        Number of messages to process in parallel (default: 8)
    reasoning_effort : str
        Reasoning effort level: 'low', 'medium', 'high' (default: 'low')
    max_completion_tokens : int
        Maximum tokens for completion (default: 2000)
    use_full_dataset : bool
        If True, use both train and test data; if False, use only test data (default: True)
    return_dataframe : bool
        If True, return (metrics, dataframe); if False, return only metrics (default: False)
    
    Returns:
    --------
    dict or tuple
        Metrics dict with accuracy, f1_score, precision, recall, confusion_matrix
        If return_dataframe=True, returns (metrics, df_results)
    
    Example:
    --------
    # Test different prompts with the same model
    metrics = evaluate_prompt_and_model('monetary_agreement_simple', 'gpt-5')
    metrics = evaluate_prompt_and_model('monetary_agreement_few_shot', 'gpt-5')
    
    # Test same prompt with different models
    metrics = evaluate_prompt_and_model('fiscal_stance_simple', 'gpt-5')
    metrics = evaluate_prompt_and_model('fiscal_stance_simple', 'gpt-5-nano')
    """
    
    print(f"\n{'='*80}")
    print(f"Evaluation: {prompt_key} with {model_name}")
    print(f"{'='*80}\n")
    
    # Step 1: Determine task type from prompt_key
    task_type = None
    for prefix in ['monetary_stance', 'monetary_agreement', 'fiscal_stance', 'fiscal_agreement']:
        if prompt_key.startswith(prefix):
            task_type = prefix
            break
    
    if task_type is None:
        raise ValueError(f"Unknown prompt type for key: {prompt_key}")
    
    print(f"Task type: {task_type}")
    
    # Step 2: Load appropriate data and define column mappings
    # data_dir_monetary = Path('/data/home/xiong/data/Fund/CSR/Traction/output/finetuning/monetary/cv')

    df_train, df_test, column_mapping, ground_truth_cols = load_transform_data(data_dir,task_type)
    # Combine datasets
    if use_full_dataset:
        sample_df = pd.concat([df_train, df_test])
        print(f"Using full dataset: {len(df_train)} train + {len(df_test)} test = {len(sample_df)} total")
    else:
        sample_df = df_test
        print(f"Using test dataset only: {len(sample_df)} samples")
    
    # Step 3: Load prompt template
    registry_entry = PROMPT_REGISTRY[prompt_key]
    prompt_file = registry_entry["prompt_file"]
    prompt_dir = Path('../../src/Traction/prompts')
    prompt_template = load_prompt(prompt_dir / prompt_file).sections
    response_model = registry_entry["response_model"]
    
    print(f"Loaded prompt: {prompt_file}")
    print(f"Response model: {response_model.__name__}")
    
    # Step 4: Build batch messages
    print(f"\nBuilding batch messages...")
    batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
        sample_df,
        prompt_template,
        column_mapping=column_mapping,
        id_column='index',
        max_text_length=8000
    )
    print(f"Generated {len(batch_messages)} batch messages")
    
    # Step 5: Initialize LLM agent and process batch
    print(f"\nProcessing with {model_name} (temperature={temperature}, batch_size={batch_size})...")
    batch_agent = BatchAsyncLLMAgent(
        api_key=os.getenv('OPENAI_API_KEY'),
        model=model_name,
        temperature=temperature,
    )
    
    async def run_batch():
        results = await _process_batch_async(
            batch_agent,
            batch_messages,
            response_model=response_model,
            batch_size=batch_size,
            max_completion_tokens=max_completion_tokens,
            reasoning_effort=reasoning_effort
        )
        return results
    
    results = asyncio.run(run_batch())
    
    # Step 6: Merge results with original data
    print(f"\nMerging results...")
    merged = _merge_ids_with_responses(batch_ids, results)
    df_merged = pd.DataFrame(merged)
    
    # Add _llm suffix to LLM-generated columns
    llm_columns = [col for col in df_merged.columns if col not in ['id']]
    rename_dict = {col: f"{col}_llm" for col in llm_columns}
    df_merged = df_merged.rename(columns=rename_dict)
    
    # Merge with original dataframe
    df_merged['id'] = df_merged['id'].astype(int)
    df_merged = df_merged.merge(sample_df, left_on='id', right_on='index', how='left')
    
    # Keep only records with no NaN values in key columns
    df_merged = df_merged.dropna(subset=[ground_truth_cols[0]])
    print(f"Valid samples for evaluation: {len(df_merged)}")
    
    # Step 7: Calculate metrics
    print(f"\n{'='*80}")
    print(f"Evaluation Metrics")
    print(f"{'='*80}\n")
    
    metrics = {}
    
    if 'agreement' in task_type:
        # Agreement task: evaluate agreement classification
        y_true = df_merged['agreement_general']
        y_pred = df_merged['agreement_llm']
        
        labels = ['disagreement exists', 'mostly agree', 'irrelevant']
        
        metrics['accuracy'] = accuracy_score(y_true, y_pred)
        metrics['f1_weighted'] = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['f1_macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)
        metrics['precision_weighted'] = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['recall_weighted'] = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred, labels=labels)
        
        print(f"Accuracy:           {metrics['accuracy']:.4f}")
        print(f"F1 (weighted):      {metrics['f1_weighted']:.4f}")
        print(f"F1 (macro):         {metrics['f1_macro']:.4f}")
        print(f"Precision (weighted): {metrics['precision_weighted']:.4f}")
        print(f"Recall (weighted):  {metrics['recall_weighted']:.4f}")
        print(f"\nConfusion Matrix:")
        print(f"Labels: {labels}")
        print(metrics['confusion_matrix'])
        
    else:
        # Stance task: evaluate multiple stance classifications
        stance_fields = ['stance_current', 'stance_future']
        
        for field in stance_fields:
            if field in df_merged.columns and f"{field}_llm" in df_merged.columns:
                y_true = df_merged[field]
                y_pred = df_merged[f"{field}_llm"]
                
                # Filter out any NaN values
                mask = y_true.notna() & y_pred.notna()
                y_true_clean = y_true[mask]
                y_pred_clean = y_pred[mask]
                
                if len(y_true_clean) > 0:
                    acc = accuracy_score(y_true_clean, y_pred_clean)
                    f1_w = f1_score(y_true_clean, y_pred_clean, average='weighted', zero_division=0)
                    f1_m = f1_score(y_true_clean, y_pred_clean, average='macro', zero_division=0)
                    
                    metrics[f'{field}_accuracy'] = acc
                    metrics[f'{field}_f1_weighted'] = f1_w
                    metrics[f'{field}_f1_macro'] = f1_m
                    
                    print(f"\n{field}:")
                    print(f"  Accuracy:      {acc:.4f}")
                    print(f"  F1 (weighted): {f1_w:.4f}")
                    print(f"  F1 (macro):    {f1_m:.4f}")
        
        # Calculate average metrics across all stance fields
        avg_acc = sum([metrics[k] for k in metrics if 'accuracy' in k]) / len([k for k in metrics if 'accuracy' in k])
        avg_f1_w = sum([metrics[k] for k in metrics if 'f1_weighted' in k]) / len([k for k in metrics if 'f1_weighted' in k])
        avg_f1_m = sum([metrics[k] for k in metrics if 'f1_macro' in k]) / len([k for k in metrics if 'f1_macro' in k])
        
        metrics['average_accuracy'] = avg_acc
        metrics['average_f1_weighted'] = avg_f1_w
        metrics['average_f1_macro'] = avg_f1_m
        
        print(f"\n{'='*40}")
        print(f"Average Metrics:")
        print(f"  Accuracy:      {avg_acc:.4f}")
        print(f"  F1 (weighted): {avg_f1_w:.4f}")
        print(f"  F1 (macro):    {avg_f1_m:.4f}")
    
    print(f"\n{'='*80}\n")
    
    if return_dataframe:
        return metrics, df_merged
    else:
        return metrics

In [9]:
def compare_evaluations(results_dict: dict, sort_by: str = 'accuracy'):
    """
    Compare multiple evaluation results in a table format.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary with format: {name: metrics_dict}
        Example: {'simple_gpt5': metrics1, 'few_shot_gpt5': metrics2}
    sort_by : str
        Metric to sort by (default: 'accuracy')
    
    Returns:
    --------
    pd.DataFrame
        Comparison table
    """
    comparison_data = []
    
    for name, metrics in results_dict.items():
        row = {'name': name}
        
        # Add relevant metrics (exclude confusion matrix as it's too large)
        for key, value in metrics.items():
            if key != 'confusion_matrix' and isinstance(value, (int, float)):
                row[key] = value
        
        comparison_data.append(row)
    
    df_comparison = pd.DataFrame(comparison_data)
    
    # Sort by specified metric if it exists
    if sort_by in df_comparison.columns:
        df_comparison = df_comparison.sort_values(by=sort_by, ascending=False)
    
    return df_comparison

## Run one example 

In [4]:
metrics,res_df = evaluate_prompt_and_model('monetary_agreement_few_shot', 'gpt-5',
                                           use_full_dataset=True,return_dataframe=True)
print(metrics)


Evaluation: monetary_agreement_few_shot with gpt-5

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_few_shot.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [04:36<00:00,  1.04msg/s]


Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7336
F1 (weighted):      0.7205
F1 (macro):         0.5660
Precision (weighted): 0.7378
Recall (weighted):  0.7336

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 34  28   1]
 [ 30 173   0]
 [  1  17   5]]


{'accuracy': 0.7335640138408305, 'f1_weighted': 0.720539408290044, 'f1_macro': 0.5659767725994485, 'precision_weighted': 0.737774543583883, 'recall_weighted': 0.7335640138408305, 'confusion_matrix': array([[ 34,  28,   1],
       [ 30, 173,   0],
       [  1,  17,   5]])}


In [ ]:
metrics,res_df = evaluate_prompt_and_model('monetary_stance_simple', 
                                           'gpt-5',
                                           use_full_dataset=True,
                                           return_dataframe=True)
print(metrics)


Evaluation: monetary_stance_simple with gpt-5-nano

Task type: monetary_stance
Using test dataset only: 58 samples
Loaded prompt: monetary_stance_simple.md
Response model: MonetaryStanceResponse

Building batch messages...
Generated 58 batch messages

Processing with gpt-5-nano (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 58/58 [00:31<00:00,  1.86msg/s]


Merging results...
Valid samples for evaluation: 58

Evaluation Metrics


stance_current:
  Accuracy:      0.6897
  F1 (weighted): 0.6855
  F1 (macro):    0.4673

stance_future:
  Accuracy:      0.6034
  F1 (weighted): 0.6410
  F1 (macro):    0.3408

Average Metrics:
  Accuracy:      0.6466
  F1 (weighted): 0.6633
  F1 (macro):    0.4040


{'stance_current_accuracy': 0.6896551724137931, 'stance_current_f1_weighted': 0.6855398092649664, 'stance_current_f1_macro': 0.4672699976542341, 'stance_future_accuracy': 0.603448275862069, 'stance_future_f1_weighted': 0.6409613375130616, 'stance_future_f1_macro': 0.34075448361162647, 'average_accuracy': 0.646551724137931, 'average_f1_weighted': 0.6632505733890139, 'average_f1_macro': 0.4040122406329303}


## Run all prompt variations

- Model : GPT-5

In [ ]:
## run all results 
results = {}
results_df = {}
# Test all monetary agreement prompts
for variant in ['simple', 'with_definitions', 'few_shot', 'chain_of_thought']:
    prompt_key = f'monetary_agreement_{variant}'
    print(f"\n\nTesting: {prompt_key}")
    metrics,res_df = evaluate_prompt_and_model(
        prompt_key=prompt_key,
        model_name='gpt-5',
        use_full_dataset=True,
        return_dataframe=True
    )
    results[f'agreement_{variant}'] = metrics
    results_df[f'agreement_{variant}'] = res_df

# Compare results
comparison_df = compare_evaluations(results, sort_by='accuracy')
print("\n\nComparison of Prompt Strategies:")
print(comparison_df)



Testing: monetary_agreement_simple

Evaluation: monetary_agreement_simple with gpt-5

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_simple.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [05:07<00:00,  1.06s/msg]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7405
F1 (weighted):      0.7241
F1 (macro):         0.5548
Precision (weighted): 0.7416
Recall (weighted):  0.7405

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 35  27   1]
 [ 28 175   0]
 [  1  18   4]]




Testing: monetary_agreement_with_definitions

Evaluation: monetary_agreement_with_definitions with gpt-5

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_with_definitions.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [05:14<00:00,  1.09s/msg]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7301
F1 (weighted):      0.7046
F1 (macro):         0.4863
Precision (weighted): 0.7117
Recall (weighted):  0.7301

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 36  26   1]
 [ 29 174   0]
 [  1  21   1]]




Testing: monetary_agreement_few_shot

Evaluation: monetary_agreement_few_shot with gpt-5

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_few_shot.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [04:53<00:00,  1.02s/msg]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7474
F1 (weighted):      0.7327
F1 (macro):         0.5810
Precision (weighted): 0.7601
Recall (weighted):  0.7474

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 35  28   0]
 [ 27 176   0]
 [  1  17   5]]




Testing: monetary_agreement_chain_of_thought

Evaluation: monetary_agreement_chain_of_thought with gpt-5

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_chain_of_thought.md
Response model: MonetaryAgreementChainOfThoughtResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [06:05<00:00,  1.26s/msg]


Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7439
F1 (weighted):      0.7242
F1 (macro):         0.5357
Precision (weighted): 0.7309
Recall (weighted):  0.7439

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 35  26   2]
 [ 26 177   0]
 [  1  19   3]]




- Model : GPT-5-nano

In [ ]:
## run all results 
results = {}
results_df = {}
# Test all monetary agreement prompts
for variant in ['simple', 'with_definitions', 'few_shot', 'chain_of_thought']:
    prompt_key = f'monetary_agreement_{variant}'
    print(f"\n\nTesting: {prompt_key}")
    metrics,res_df = evaluate_prompt_and_model(
        prompt_key=prompt_key,
        model_name='gpt-5-nano',
        use_full_dataset=True,
        return_dataframe=True
    )
    results[f'agreement_{variant}'] = metrics
    results_df[f'agreement_{variant}'] = res_df



Testing: monetary_agreement_simple

Evaluation: monetary_agreement_simple with gpt-5-nano

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_simple.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-nano (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [02:10<00:00,  2.22msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.6574
F1 (weighted):      0.6354
F1 (macro):         0.4026
Precision (weighted): 0.6160
Recall (weighted):  0.6574

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 31  32   0]
 [ 44 159   0]
 [  3  20   0]]




Testing: monetary_agreement_with_definitions

Evaluation: monetary_agreement_with_definitions with gpt-5-nano

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_with_definitions.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-nano (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [02:18<00:00,  2.08msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.6367
F1 (weighted):      0.6126
F1 (macro):         0.3758
Precision (weighted): 0.5904
Recall (weighted):  0.6367

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 25  38   0]
 [ 44 159   0]
 [  3  20   0]]




Testing: monetary_agreement_few_shot

Evaluation: monetary_agreement_few_shot with gpt-5-nano

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_few_shot.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-nano (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [02:10<00:00,  2.21msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.6644
F1 (weighted):      0.6395
F1 (macro):         0.4015
Precision (weighted): 0.6167
Recall (weighted):  0.6644

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 29  34   0]
 [ 40 163   0]
 [  4  19   0]]




Testing: monetary_agreement_chain_of_thought

Evaluation: monetary_agreement_chain_of_thought with gpt-5-nano

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_chain_of_thought.md
Response model: MonetaryAgreementChainOfThoughtResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-nano (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [02:26<00:00,  1.98msg/s]


Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.6298
F1 (weighted):      0.6136
F1 (macro):         0.4012
Precision (weighted): 0.6692
Recall (weighted):  0.6298

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 26  37   0]
 [ 48 155   0]
 [  3  19   1]]




In [8]:
# Compare results
comparison_df = compare_evaluations(results, sort_by='accuracy')
print("\n\nComparison of Prompt Strategies:")
print(comparison_df)



Comparison of Prompt Strategies:
                         name  accuracy  f1_weighted  f1_macro  \
2          agreement_few_shot  0.664360     0.639482  0.401505   
0            agreement_simple  0.657439     0.635397  0.402611   
1  agreement_with_definitions  0.636678     0.612572  0.375838   
3  agreement_chain_of_thought  0.629758     0.613569  0.401185   

   precision_weighted  recall_weighted  
2            0.616669         0.664360  
0            0.615952         0.657439  
1            0.590370         0.636678  
3            0.669190         0.629758  


In [9]:
comparison_df

,name,accuracy,f1_weighted,f1_macro,precision_weighted,recall_weighted
2,agreement_few_shot,0.664360,0.639482,0.401505,0.616669,0.664360
0,agreement_simple,0.657439,0.635397,0.402611,0.615952,0.657439
1,agreement_with_definitions,0.636678,0.612572,0.375838,0.590370,0.636678
3,agreement_chain_of_thought,0.629758,0.613569,0.401185,0.669190,0.629758


- Model: GPT-5-mini

In [10]:
## run all results 
results = {}
results_df = {}
# Test all monetary agreement prompts
for variant in ['simple', 'with_definitions', 'few_shot', 'chain_of_thought']:
    prompt_key = f'monetary_agreement_{variant}'
    print(f"\n\nTesting: {prompt_key}")
    metrics,res_df = evaluate_prompt_and_model(
        prompt_key=prompt_key,
        model_name='gpt-5-mini',
        use_full_dataset=True,
        return_dataframe=True
    )
    results[f'agreement_{variant}'] = metrics
    results_df[f'agreement_{variant}'] = res_df
## agg results    
comparison_df = compare_evaluations(results, sort_by='accuracy')
print("\n\nComparison of Prompt Strategies:")
comparison_df



Testing: monetary_agreement_simple

Evaluation: monetary_agreement_simple with gpt-5-mini

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_simple.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-mini (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [04:20<00:00,  1.11msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7059
F1 (weighted):      0.6774
F1 (macro):         0.4390
Precision (weighted): 0.6511
Recall (weighted):  0.7059

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 34  29   0]
 [ 33 170   0]
 [  3  20   0]]




Testing: monetary_agreement_with_definitions

Evaluation: monetary_agreement_with_definitions with gpt-5-mini

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_with_definitions.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-mini (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [03:59<00:00,  1.21msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7093
F1 (weighted):      0.6837
F1 (macro):         0.4492
Precision (weighted): 0.6609
Recall (weighted):  0.7093

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 38  25   0]
 [ 36 167   0]
 [  3  20   0]]




Testing: monetary_agreement_few_shot

Evaluation: monetary_agreement_few_shot with gpt-5-mini

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_few_shot.md
Response model: MonetaryAgreementResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-mini (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [03:16<00:00,  1.47msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7197
F1 (weighted):      0.6944
F1 (macro):         0.4770
Precision (weighted): 0.7410
Recall (weighted):  0.7197

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 35  28   0]
 [ 31 172   0]
 [  2  20   1]]




Testing: monetary_agreement_chain_of_thought

Evaluation: monetary_agreement_chain_of_thought with gpt-5-mini

Task type: monetary_agreement
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_agreement_chain_of_thought.md
Response model: MonetaryAgreementChainOfThoughtResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-mini (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [04:59<00:00,  1.04s/msg]


Merging results...
Valid samples for evaluation: 289

Evaluation Metrics

Accuracy:           0.7163
F1 (weighted):      0.6945
F1 (macro):         0.4799
Precision (weighted): 0.7452
Recall (weighted):  0.7163

Confusion Matrix:
Labels: ['disagreement exists', 'mostly agree', 'irrelevant']
[[ 38  25   0]
 [ 35 168   0]
 [  3  19   1]]




Comparison of Prompt Strategies:


,name,accuracy,f1_weighted,f1_macro,precision_weighted,recall_weighted
2,agreement_few_shot,0.719723,0.694354,0.476974,0.740953,0.719723
3,agreement_chain_of_thought,0.716263,0.694531,0.479911,0.745218,0.716263
1,agreement_with_definitions,0.709343,0.683662,0.449225,0.660904,0.709343
0,agreement_simple,0.705882,0.677388,0.438988,0.651142,0.705882


### Try different task

#### Monetary Stance Task

- Model: GPT-5

In [ ]:
## run all results 
results = {}
results_df = {}
# Test all monetary agreement prompts
for variant in ['simple', 'with_definitions', 'few_shot', 'chain_of_thought']:
    prompt_key = f'monetary_stance_{variant}'
    print(f"\n\nTesting: {prompt_key}")
    metrics,res_df = evaluate_prompt_and_model(
        prompt_key=prompt_key,
        model_name='gpt-5',
        use_full_dataset=True,
        return_dataframe=True
    )
    results[f'stance_{variant}'] = metrics
    results_df[f'stance_{variant}'] = res_df




Testing: monetary_stance_simple

Evaluation: monetary_stance_simple with gpt-5

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_simple.md
Response model: MonetaryStanceResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [04:47<00:00,  1.00msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics


stance_current:
  Accuracy:      0.7405
  F1 (weighted): 0.7249
  F1 (macro):    0.5363

stance_future:
  Accuracy:      0.6609
  F1 (weighted): 0.6691
  F1 (macro):    0.4863

Average Metrics:
  Accuracy:      0.7007
  F1 (weighted): 0.6970
  F1 (macro):    0.5113




Testing: monetary_stance_with_definitions

Evaluation: monetary_stance_with_definitions with gpt-5

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_with_definitions.md
Response model: MonetaryStanceResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [05:02<00:00,  1.05s/msg]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics


stance_current:
  Accuracy:      0.6505
  F1 (weighted): 0.6091
  F1 (macro):    0.4069

stance_future:
  Accuracy:      0.6228
  F1 (weighted): 0.6417
  F1 (macro):    0.4233

Average Metrics:
  Accuracy:      0.6367
  F1 (weighted): 0.6254
  F1 (macro):    0.4151




Testing: monetary_stance_few_shot

Evaluation: monetary_stance_few_shot with gpt-5

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_few_shot.md
Response model: MonetaryStanceResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [04:25<00:00,  1.09msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics


stance_current:
  Accuracy:      0.7855
  F1 (weighted): 0.7747
  F1 (macro):    0.6840

stance_future:
  Accuracy:      0.6955
  F1 (weighted): 0.6981
  F1 (macro):    0.5571

Average Metrics:
  Accuracy:      0.7405
  F1 (weighted): 0.7364
  F1 (macro):    0.6206




Testing: monetary_stance_chain_of_thought

Evaluation: monetary_stance_chain_of_thought with gpt-5

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_chain_of_thought.md
Response model: MonetaryStanceChainOfThoughtResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5 (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [05:34<00:00,  1.16s/msg]


Merging results...
Valid samples for evaluation: 289

Evaluation Metrics


stance_current:
  Accuracy:      0.7266
  F1 (weighted): 0.7068
  F1 (macro):    0.4808

stance_future:
  Accuracy:      0.6817
  F1 (weighted): 0.6815
  F1 (macro):    0.4696

Average Metrics:
  Accuracy:      0.7042
  F1 (weighted): 0.6942
  F1 (macro):    0.4752




NameError: name 'compare_evaluations' is not defined

In [10]:
## agg results    
comparison_df = compare_evaluations(results, sort_by='accuracy')
print("\n\nComparison of Prompt Strategies:")
comparison_df



Comparison of Prompt Strategies:


,name,stance_current_accuracy,stance_current_f1_weighted,stance_current_f1_macro,stance_future_accuracy,stance_future_f1_weighted,stance_future_f1_macro,average_accuracy,average_f1_weighted,average_f1_macro
0,stance_simple,0.740484,0.724939,0.536349,0.660900,0.669137,0.486256,0.700692,0.697038,0.511303
1,stance_with_definitions,0.650519,0.609066,0.406939,0.622837,0.641722,0.423338,0.636678,0.625394,0.415138
2,stance_few_shot,0.785467,0.774659,0.684032,0.695502,0.698075,0.557122,0.740484,0.736367,0.620577
3,stance_chain_of_thought,0.726644,0.706836,0.480777,0.681661,0.681472,0.469626,0.704152,0.694154,0.475202


- Model: GPT-5-mini

In [ ]:
## run all results 
results = {}
results_df = {}
# Test all monetary agreement prompts
for variant in ['simple', 'with_definitions', 'few_shot', 'chain_of_thought']:
    prompt_key = f'monetary_stance_{variant}'
    print(f"\n\nTesting: {prompt_key}")
    metrics,res_df = evaluate_prompt_and_model(
        prompt_key=prompt_key,
        model_name='gpt-5-mini',
        use_full_dataset=True,
        return_dataframe=True
    )
    results[f'stance_{variant}'] = metrics
    results_df[f'stance_{variant}'] = res_df
## agg results    
comparison_df = compare_evaluations(results, sort_by='accuracy')
print("\n\nComparison of Prompt Strategies:")
comparison_df



Testing: monetary_stance_simple

Evaluation: monetary_stance_simple with gpt-5-mini

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_simple.md
Response model: MonetaryStanceResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-mini (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [03:50<00:00,  1.25msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics


stance_current:
  Accuracy:      0.6471
  F1 (weighted): 0.6158
  F1 (macro):    0.4065

stance_future:
  Accuracy:      0.6298
  F1 (weighted): 0.6237
  F1 (macro):    0.3585

Average Metrics:
  Accuracy:      0.6384
  F1 (weighted): 0.6198
  F1 (macro):    0.3825




Testing: monetary_stance_with_definitions

Evaluation: monetary_stance_with_definitions with gpt-5-mini

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_with_definitions.md
Response model: MonetaryStanceResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-mini (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [03:47<00:00,  1.27msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics


stance_current:
  Accuracy:      0.6125
  F1 (weighted): 0.5566
  F1 (macro):    0.3663

stance_future:
  Accuracy:      0.5813
  F1 (weighted): 0.5961
  F1 (macro):    0.3769

Average Metrics:
  Accuracy:      0.5969
  F1 (weighted): 0.5764
  F1 (macro):    0.3716




Testing: monetary_stance_few_shot

Evaluation: monetary_stance_few_shot with gpt-5-mini

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_few_shot.md
Response model: MonetaryStanceResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-mini (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [03:13<00:00,  1.49msg/s]



Merging results...
Valid samples for evaluation: 289

Evaluation Metrics


stance_current:
  Accuracy:      0.6747
  F1 (weighted): 0.6386
  F1 (macro):    0.4654

stance_future:
  Accuracy:      0.6332
  F1 (weighted): 0.6376
  F1 (macro):    0.4288

Average Metrics:
  Accuracy:      0.6540
  F1 (weighted): 0.6381
  F1 (macro):    0.4471




Testing: monetary_stance_chain_of_thought

Evaluation: monetary_stance_chain_of_thought with gpt-5-mini

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_chain_of_thought.md
Response model: MonetaryStanceChainOfThoughtResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-mini (temperature=1.0, batch_size=8)...


Processing batches: 100%|██████████| 289/289 [04:21<00:00,  1.11msg/s]


Merging results...
Valid samples for evaluation: 289

Evaluation Metrics


stance_current:
  Accuracy:      0.6401
  F1 (weighted): 0.5944
  F1 (macro):    0.4311

stance_future:
  Accuracy:      0.6228
  F1 (weighted): 0.6248
  F1 (macro):    0.4172

Average Metrics:
  Accuracy:      0.6315
  F1 (weighted): 0.6096
  F1 (macro):    0.4241




Comparison of Prompt Strategies:


,name,stance_current_accuracy,stance_current_f1_weighted,stance_current_f1_macro,stance_future_accuracy,stance_future_f1_weighted,stance_future_f1_macro,average_accuracy,average_f1_weighted,average_f1_macro
0,agreement_simple,0.647059,0.615826,0.406451,0.629758,0.623676,0.358488,0.638408,0.619751,0.382470
1,agreement_with_definitions,0.612457,0.556638,0.366253,0.581315,0.596068,0.376922,0.596886,0.576353,0.371588
2,agreement_few_shot,0.674740,0.638552,0.465397,0.633218,0.637611,0.428846,0.653979,0.638082,0.447122
3,agreement_chain_of_thought,0.640138,0.594402,0.431085,0.622837,0.624814,0.417192,0.631488,0.609608,0.424138


- Model: GPT-5-nano

In [ ]:
## run all results 
results = {}
results_df = {}
# Test all monetary agreement prompts
for variant in ['simple', 'with_definitions', 'few_shot', 'chain_of_thought']:
    prompt_key = f'monetary_stance_{variant}'
    print(f"\n\nTesting: {prompt_key}")
    metrics,res_df = evaluate_prompt_and_model(
        prompt_key=prompt_key,
        model_name='gpt-5-nano',
        use_full_dataset=True,
        return_dataframe=True
    )
    results[f'stance_{variant}'] = metrics
    results_df[f'stance_{variant}'] = res_df
## agg results    
comparison_df = compare_evaluations(results, sort_by='accuracy')
print("\n\nComparison of Prompt Strategies:")
comparison_df



Testing: monetary_stance_simple

Evaluation: monetary_stance_simple with gpt-5-nano

Task type: monetary_stance
Using full dataset: 231 train + 58 test = 289 total
Loaded prompt: monetary_stance_simple.md
Response model: MonetaryStanceResponse

Building batch messages...
Generated 289 batch messages

Processing with gpt-5-nano (temperature=1.0, batch_size=8)...


Processing batches:  58%|█████▊    | 168/289 [01:31<01:03,  1.89msg/s]